# Scripting no Processamento de Linguagem Natural - TP2
## Information Retrieval Pipeline - Movie Articles Corpus

---
### Imports and Setup

In [1]:
import json
import re
import math
import torch
import random
import string
import warnings
from datasets import Dataset, load_dataset
from pathlib import Path
from itertools import combinations
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainingArguments, SentenceTransformerTrainer, util
from sentence_transformers.sentence_transformer.losses import CosineSimilarityLoss
from sentence_transformers.sentence_transformer.evaluation import EmbeddingSimilarityEvaluator, SimilarityFunction

from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM, pipeline

import unicodedata
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')

STOPWORDS = set(stopwords.words("english"))
LEMMATIZER = WordNetLemmatizer()

CORPUS_PATH = 'articles.json'
   
print('Setup complete.')

/home/duartearaujo/Desktop/MEI/SPLN/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setup complete.


[nltk_data] Downloading package stopwords to
[nltk_data]     /home/duartearaujo/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /home/duartearaujo/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /home/duartearaujo/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


---
### Text Cleaning and Preprocessing

In [2]:
def clean_text(text):
    if not text:
        return ""

    text = unicodedata.normalize("NFKD", text)

    text = re.sub(r"==+.*?==+", " ", text)
    text = re.sub(r"http\\S+|www\\.\\S+", " ", text)
    text = re.sub(r"\\[\\d+\\]", " ", text)
    text = re.sub(r"[^a-zA-Z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()

    text = text.lower()

    return text

def extract_intro(text):
    parts = re.split(r"==+.*?==+", text)
    intro = parts[0].strip() if parts else text[:1500]

    return intro

def extract_see_also(text):
    match = re.search(r"==+\s*See also\s*==+(.*?)(==+|$)", text, re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(1).strip()
    return ""

def preprocess_tokens(text):
    tokens = word_tokenize(text)

    processed = []

    for token in tokens:
        if not token.isalpha():
            continue
        if token in STOPWORDS:
            continue
        if len(token) < 3:
            continue

        lemma = LEMMATIZER.lemmatize(token)
        processed.append(lemma)

    return processed

def chunkify(text: str, chunk_size: int = 5000, stride: int = 2500) -> list[str]:
    chunks, start = [], 0
    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunks.append(text[start:end])
        if end == len(text):
            break
        start += chunk_size - stride
    return chunks

def preprocess_document(doc):
    raw_text = doc["script"]

    intro = extract_intro(raw_text)
    see_also = extract_see_also(raw_text)
    cleaned_text = clean_text(raw_text)
    cleaned_intro = clean_text(intro)
    tokens = preprocess_tokens(cleaned_text)

    return {
        "title": doc["title"],
        "intro": cleaned_intro,
        "text": cleaned_text,
        "tokens": tokens,
        "see_also": see_also,
        "genres": doc.get("genres", []),
        "chunks": chunkify(cleaned_text)
    }

with open("articles.json", "r", encoding="utf-8") as f:
    corpus = json.load(f)

processed_corpus = [
    preprocess_document(doc)
    for doc in corpus
]

with open("processed_articles.json", "w", encoding="utf-8") as f:
    json.dump(processed_corpus, f, ensure_ascii=False, indent=1)

print(f"Processed {len(processed_corpus)} articles.")
print(f"Total tokens in corpus: {sum(len(doc['tokens']) for doc in processed_corpus)}")
print(f"Average tokens per article: {sum(len(doc['tokens']) for doc in processed_corpus) / len(processed_corpus):.2f}")
print(f"Total chunks in corpus: {sum(len(doc['chunks']) for doc in processed_corpus)}")
print(f"Average chunks per article: {sum(len(doc['chunks']) for doc in processed_corpus) / len(processed_corpus):.2f}")

Processed 95 articles.
Total tokens in corpus: 401886
Average tokens per article: 4230.38
Total chunks in corpus: 1677
Average chunks per article: 17.65


In [4]:
all_tokens = [t for doc in processed_corpus for t in doc['tokens']]
vocab = Counter(all_tokens)
top30 = vocab.most_common(30)

print(f'Unique vocabulary size: {len(vocab):,}')
print('Top 30 most common tokens:')
for token, count in top30:
    print(f'  {token}: {count:,}')

Unique vocabulary size: 29,306
Top 30 most common tokens:
  film: 12,186
  first: 1,867
  time: 1,780
  one: 1,711
  also: 1,585
  studio: 1,568
  new: 1,568
  year: 1,493
  picture: 1,317
  production: 1,278
  would: 1,251
  director: 1,244
  movie: 1,236
  best: 1,185
  two: 1,153
  award: 1,116
  made: 1,088
  american: 1,037
  scene: 1,027
  work: 964
  star: 898
  released: 891
  later: 873
  war: 854
  many: 817
  feature: 807
  actor: 796
  release: 791
  sound: 786
  company: 786


---
### Lexical Retrieval — BM25

In [5]:
tokenized_corpus = [doc['tokens'] for doc in processed_corpus]
bm25 = BM25Okapi(tokenized_corpus, k1=1.5, b=0.75)
print(f'BM25 index built: {len(tokenized_corpus)} documents, k1=1.5, b=0.75')

BM25 index built: 95 documents, k1=1.5, b=0.75


In [ ]:
def bm25_retrieve(query: str, top_k: int = 5) -> list:
    q_tokens = preprocess_tokens(query)
    if not q_tokens:
        return []
    
    scores = bm25.get_scores(q_tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]

    return [
        {**{k: processed_corpus[i][k] for k in ('title','see_also')},
         'bm25_score': round(float(scores[i]), 4),
         'doc_idx': int(i)}
        for i in top_idx
    ]

demo_queries = [
    "Technicolor process",
    "History of silent film",
    "Italian neorealism",
    "samurai japanese cinema",
    "Origins of sound film",
    "Charlie Chaplin slapstick",
    "horror shark summer blockbuster",
    "computer animation in film",
    "film noir characteristics",
]

for q in demo_queries:
    res = bm25_retrieve(q, top_k=5)
    print(f'\nQuery: "{q}"')
    for i, r in enumerate(res, 1):
        print(f'  {i}. {r["title"]:<45} {r["genres"]}  score={r["bm25_score"]}')


Query: "Technicolor process"
  1. Technicolor                                   ['encyclopedia']  score=2.0648
  2. CinemaScope                                   ['encyclopedia']  score=2.0102
  3. Lumière brothers                              ['encyclopedia']  score=1.9659
  4. IMAX                                          ['encyclopedia']  score=1.8682
  5. Thomas Edison                                 ['encyclopedia']  score=1.8508

Query: "History of silent film"
  1. Silent film                                   ['encyclopedia']  score=2.4466
  2. Sound film                                    ['encyclopedia']  score=2.4223
  3. Harold Lloyd                                  ['encyclopedia']  score=2.4138
  4. Lillian Gish                                  ['encyclopedia']  score=2.4068
  5. Buster Keaton                                 ['encyclopedia']  score=2.4035

Query: "Italian neorealism"
  1. Italian neorealism                            ['encyclopedia']  score=6.0072
  2. B

---
### Semantic Retrieval — Sentence-BERT

In [7]:
def see_score(see_also: list, see_also2: list) -> float:
    overlap = len(set(see_also) & set(see_also2))
    if overlap == 0:   return 0.0
    if overlap > 0 and overlap < 20:   return 0.4
    return 0.7

representative = [
    {
        'title':  doc['title'],
        'see_also': doc['see_also'],
        'text': doc['text'],
    }
    for doc in processed_corpus
]

cross_pairs = []
for doc1, doc2 in combinations(representative, 2):
    score = see_score(doc1['see_also'], doc2['see_also'])
    cross_pairs.append((doc1['text'], doc2['text'], score))

print(f'Cross-document pairs: {len(cross_pairs):,}')
print(f'\nCross-pair score distribution:')
print(Counter(round(score, 1) for _, _, score in cross_pairs))

Cross-document pairs: 4,465

Cross-pair score distribution:
Counter({0.0: 1837, 0.4: 1497, 0.7: 1131})


In [ ]:
"""
majority = [p for p in cross_pairs if p[2] == 0.0]
minority = [p for p in cross_pairs if p[2] == 0.7]

majority_balanced = resample(
    majority,
    replace=False,
    n_samples=len(minority),
    random_state=42,
)

all_pairs = majority_balanced + minority 
random.shuffle(all_pairs)

print(f'Balanced cross pairs: {len(majority_balanced) + len(minority):,}')
print(f'Total pairs:          {len(all_pairs):,}')
print(f'\nFinal score distribution:')
print(Counter(round(s, 1) for _, _, s in all_pairs))
"""

Balanced cross pairs: 2,262
Total pairs:          2,262

Final score distribution:
Counter({0.0: 1131, 0.7: 1131})


In [ ]:
scores = [s for _, _, s in cross_pairs]

train_data, test_data = train_test_split(
    cross_pairs,
    test_size=0.2,
    random_state=42,
    stratify=[round(s, 1) for s in scores],
)

print('Train distribution:', Counter(round(s, 1) for _, _, s in train_data))
print('Test  distribution:', Counter(round(s, 1) for _, _, s in test_data))

train_dataset = Dataset.from_list([
    {'text1': p1, 'text2': p2, 'score': float(score)}
    for p1, p2, score in train_data
])

test_dataset = Dataset.from_list([
    {'text1': p1, 'text2': p2, 'score': float(score)}
    for p1, p2, score in test_data
])

print(train_dataset)
print(test_dataset)

Train distribution: Counter({0.7: 905, 0.0: 904})
Test  distribution: Counter({0.0: 227, 0.7: 226})
Dataset({
    features: ['text1', 'text2', 'score'],
    num_rows: 1809
})
Dataset({
    features: ['text1', 'text2', 'score'],
    num_rows: 453
})


In [10]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f'Model loaded. Embedding dimension: {model.get_embedding_dimension()}')

loss = CosineSimilarityLoss(model)

args = SentenceTransformerTrainingArguments(
    # Required parameter:
    output_dir="models/bi-encoder",
    # Optional training parameters:
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    warmup_steps=0.1,
    fp16=True,  
    bf16=False,  
    # Optional tracking/debugging parameters:
    eval_strategy="epoch",
    eval_steps=100,
    save_strategy="epoch",
    save_steps=100,
    save_total_limit=2,
    logging_steps=100,
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3314.00it/s]


Model loaded. Embedding dimension: 384


In [11]:
dev_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=test_dataset['text1'],
    sentences2=test_dataset['text2'],
    scores=test_dataset['score'],
    main_similarity=SimilarityFunction.COSINE,
    name='movie-articles-eval',
)

baseline_res = dev_evaluator(model)
print(f'Baseline Spearman correlation (before fine-tuning): {baseline_res["movie-articles-eval_spearman_cosine"]:.4f}')

Baseline Spearman correlation (before fine-tuning): 0.2647


In [12]:
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    loss=loss,
    evaluator=dev_evaluator,
)

trainer.train()

/home/duartearaujo/Desktop/MEI/SPLN/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Movie-articles-eval Pearson Cosine,Movie-articles-eval Spearman Cosine
1,0.059482,0.024148,0.896463,0.826626


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 14.80it/s]


TrainOutput(global_step=114, training_loss=0.054403214862472134, metrics={'train_runtime': 709.385, 'train_samples_per_second': 2.55, 'train_steps_per_second': 0.161, 'total_flos': 0.0, 'train_loss': 0.054403214862472134, 'epoch': 1.0})

In [14]:
finetuned_res = dev_evaluator(model)
print(f'Fine-tuned Spearman correlation: {finetuned_res["movie-articles-eval_spearman_cosine"]:.4f}')
print(f'Improvement: {finetuned_res["movie-articles-eval_spearman_cosine"] - baseline_res["movie-articles-eval_spearman_cosine"]:+.4f}')

model.save('models/sbert_movie_articles/final')
print('Model saved to models/sbert_movie_articles/final')

Fine-tuned Spearman correlation: 0.8266
Improvement: +0.5620


Writing model shards: 100%|██████████| 1/1 [00:00<00:00, 20.27it/s]


Model saved to models/sbert_movie_articles/final


In [15]:
finetuned_model = SentenceTransformer('models/sbert_movie_articles/final')

SBERT_CHAR_LIMIT = 5000
script_texts = [doc['text'][:SBERT_CHAR_LIMIT] for doc in processed_corpus]

print('Re-encoding corpus with fine-tuned model...')
finetuned_embeddings = finetuned_model.encode(
    script_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_tensor=True,
)
print(f'Embeddings shape: {finetuned_embeddings.shape}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10175.31it/s]


Re-encoding corpus with fine-tuned model...


Batches: 100%|██████████| 3/3 [00:02<00:00,  1.32it/s]

Embeddings shape: torch.Size([95, 384])


In [16]:
def sbert_retrieve_finetuned(query: str, top_k: int = 5) -> list:
    query_embedding = finetuned_model.encode(query, convert_to_tensor=True)
    cos_scores      = util.pytorch_cos_sim(query_embedding, finetuned_embeddings)[0]
    top_results     = torch.topk(cos_scores, k=top_k)
    return [
        {
            'title':       processed_corpus[int(idx)]['title'],
            'see_also':    processed_corpus[int(idx)]['see_also'],
            'sbert_score': round(float(score), 4),
        }
        for score, idx in zip(top_results.values, top_results.indices)
    ]


test_query = "the transition from silent movies to talking movies"

base_model = SentenceTransformer('all-MiniLM-L6-v2')
base_embeddings = base_model.encode(script_texts, batch_size=32, convert_to_tensor=True)

def sbert_retrieve_base(query, top_k=5):
    q_emb = base_model.encode(query, convert_to_tensor=True)
    scores = util.pytorch_cos_sim(q_emb, base_embeddings)[0]
    top = torch.topk(scores, k=top_k)
    return [{'title': processed_corpus[int(i)]['title'], 'score': round(float(s), 4)}
            for s, i in zip(top.values, top.indices)]

print(f'Query: "{test_query}"\n')
print(f'{"Rank":<5} {"Base model":<45} {"Fine-tuned"}')
print('-' * 90)
base_res = sbert_retrieve_base(test_query)
ft_res   = sbert_retrieve_finetuned(test_query)
for i, (b, f) in enumerate(zip(base_res, ft_res), 1):
    print(f'{i:<5} {b["title"]:<45} {f["title"]}')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11574.68it/s]


Query: "the transition from silent movies to talking movies"

Rank  Base model                                    Fine-tuned
------------------------------------------------------------------------------------------
1     Silent film                                   Sound film
2     Sound film                                    Silent film
3     History of film                               Ben-Hur (1959 film)
4     Cinema of the United States                   Film noir
5     Lillian Gish                                  History of animation


---
### Passage Chunking for QA

In [21]:
passages = []
for doc_idx, doc in enumerate(processed_corpus):
    for chunk in doc['chunks']:
        passages.append({
            'doc_idx': doc_idx,
            'title': doc['title'],
            'see_also': doc['see_also'],
            'chunk': chunk
        })

print(f'Total passages: {len(passages):,}')
print(f'Avg passages per script: {len(passages) / len(processed_corpus):.1f}')
print(f'\nSample passage (doc 0, passage 2):\n---')
print(passages[2]['chunk'][:400])

Total passages: 3,339
Avg passages per script: 35.1

Sample passage (doc 0, passage 2):
---
90  old traditions of charlatans and entertainers using camera obscura or magic lantern to project images of demons or ghosts were further developed into a type of multimedia show known as phantasmagoria  these popular shows entertained audiences using mechanical slides  rear projection  mobile projectors  superimposition  dissolves  live actors  smoke  on which projections may have been cast   od


In [22]:
passage_bm25   = BM25Okapi([preprocess_tokens(p['chunk']) for p in passages], k1=1.5, b=0.75)
print(f'Passage-level BM25 index built: {len(passages):,} passages.')

Passage-level BM25 index built: 3,339 passages.


In [23]:
def retrieve_passages(query: str, top_k: int = 3) -> list[dict]:
    q_tokens = preprocess_tokens(query)
    if not q_tokens:
        return []
    scores = passage_bm25.get_scores(q_tokens)
    top_idx = np.argsort(scores)[::-1][:top_k]
    results = []
    for idx in top_idx:
        results.append({
            'title':   passages[idx]['title'],
            'passage': passages[idx]['chunk'],
            'score':   scores[idx],
        })
    return results


test_results = retrieve_passages('the transition from silent movies to talking movies', top_k=5)
for r in test_results:
    print(f"\n{r['title']} (score={r['score']:.2f})")
    print(r['passage'][:100])


Singin' in the Rain (score=12.53)
singin  in the rain is a 1952 american musical romantic comedy film directed and choreographed by ge

Sound film (score=11.93)
 september  beggars of life  though it had just a few lines of dialogue  it demonstrated the studio 

Harold Lloyd (score=11.78)
that had not yet converted to sound  but the talking version became the standard edition of the film

Harold Lloyd (score=11.77)
aimed grandma s boy  which  along with chaplin s the kid  pioneered the combination of complex chara

Sound film (score=11.69)
e of treatment heretofore developed by the silent drama     the talking scenes will require differen


---
### Extractive QA

In [24]:
print('Loading extractive QA model (roberta-base-squad2)...')
qa_tokenizer = AutoTokenizer.from_pretrained('deepset/roberta-base-squad2')
qa_model     = AutoModelForQuestionAnswering.from_pretrained('deepset/roberta-base-squad2')


Loading extractive QA model (roberta-base-squad2)...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3281.66it/s]


In [25]:
def run_extractive_qa(question: str, passage: str) -> dict:
    inputs = qa_tokenizer(
        question,
        passage,
        return_tensors='pt',
        truncation=True,
        max_length=512,
    )

    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        outputs = qa_model(**inputs)

    start_idx = torch.argmax(outputs.start_logits)
    end_idx = torch.argmax(outputs.end_logits) + 1

    answer_tokens = inputs['input_ids'][0][start_idx:end_idx]

    answer = qa_tokenizer.decode(
        answer_tokens,
        skip_special_tokens=True,
    )

    score = (
        torch.max(outputs.start_logits).item() +
        torch.max(outputs.end_logits).item()
    ) / 2

    return {
        'answer': answer.strip(),
        'score': float(score),
    }

def extractive_answer(question: str, top_k_passages: int = 5) -> dict:
    retrieved = retrieve_passages(question, top_k=top_k_passages)
    if not retrieved:
        return {'answer': 'No relevant passages found.', 'score': 0.0}
    
    best = {
        'answer': None,
        'score':  -1.0,
        'source': None,
        'passage': None,
    }

    for p in retrieved:
        try:
            result = run_extractive_qa(
                question,
                p['passage'][:2000],
            )
            if result['score'] > best['score']:
                best = {
                    'answer': result['answer'],
                    'score': float(result['score']),
                    'source': p['title'],
                    'passage': p['passage'],
                }
                
        except Exception:
            continue

    return best

questions = [
    "Who directed Citizen Kane?",
    "When was The Jazz Singer released?",
    "What is considered the first science fiction film?",
    "Which movie introduced synchronized sound?",
    "Who founded Pixar?",
    "What year did Technicolor become popular?",
    "Which film is associated with the Rashomon effect?",
    "What was the first permanent movie theater called?",
    "Who directed Seven Samurai?",
    "Which company created the Kinetoscope?",
    "What movement is Bicycle Thieves associated with?",
    "Which film pioneered German Expressionism?",
    "What caused the decline of the studio system?",
    "Who created the Cinématographe?",
    "What is film noir?"
]

for q in questions:
    result = extractive_answer(q)
    print(f'\nQuestion: {q}')
    print(f'     Answer : {result["answer"]}')
    print(f'     Source : {result["source"]}')
    print(f'     Score  : {result["score"]}')
    print(f'     Context: {result["passage"][:200]}...')


Question: Who directed Citizen Kane?
     Answer : luchino visconti
     Source : Film noir
     Score  : 4.577388525009155
     Context: c period include brighton rock  1947   directed by john boulting  they made me a fugitive  1947   directed by alberto cavalcanti  the october man  1947   directed by roy ward baker  the small back roo...

Question: When was The Jazz Singer released?
     Answer : 
     Source : 3D film
     Score  : 4.871619701385498
     Context: g russia where it opened in 3d on 295 screens  on january 19  2008  u2 3d was released  it was the first live action digital 3d film  in the same year others 3d films included hannah montana   miley c...

Question: What is considered the first science fiction film?
     Answer : a trip to the moon
     Source : A Trip to the Moon
     Score  : 5.884530782699585
     Context:  component of certain types of cinema  including science fiction films  musicals  and avant garde films       with its pioneering use of themes of sci

---
### Abstractive QA

In [26]:
print('Loading abstractive QA model (flan-t5-base)...')
abs_tokenizer = AutoTokenizer.from_pretrained('google/flan-t5-base')
abs_qa_model = AutoModelForSeq2SeqLM.from_pretrained('google/flan-t5-base')
print('Abstractive model loaded.')

Loading abstractive QA model (flan-t5-base)...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1875.75it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Abstractive model loaded.


In [ ]:
def build_prompt(question: str, passages: list[dict]) -> str:
    context_parts = []
    for i, p in enumerate(passages[:3], 1):
        context_parts.append(f"[Source {i}: {p['title']}]\n{p['passage'][:600]}")
    context = '\n\n'.join(context_parts)

    prompt = (
        f"Answer the question concisely and accurately, only using the provided sources. If the answer is not present, say so."
        f"Context:\n{context}\n\n"
        f"Question: {question}\n\n"
        f"Answer:"
    )
    return prompt


def abstractive_answer(question: str, top_k_passages: int = 3) -> dict:
    retrieved = retrieve_passages(question, top_k=top_k_passages)
    if not retrieved:
        return {'answer': 'No relevant passages found.', 'sources': []}

    prompt = build_prompt(question, retrieved)
    inputs = abs_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=1024,
    )

    with torch.no_grad():
        outputs = abs_qa_model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=4,
            early_stopping=True
        )

    output = abs_tokenizer.decode(outputs[0], skip_special_tokens=True)

    return {
        'answer':  output.strip(),
        'sources': [r['title'] for r in retrieved],
        'prompt_preview': prompt[:500] + '...'
    }


for q in questions:
    result = abstractive_answer(q)
    print(f'\nQuestion: {q}')
    print(f'     Answer : {result["answer"]}')
    print(f'     Sources: {result["sources"]}')


Question: Who directed Citizen Kane?
     Answer : jules dassin
     Sources: ['Film noir', 'Sound film', 'Film noir']

Question: When was The Jazz Singer released?
     Answer : february 1 2020
     Sources: ['Studio Ghibli', 'Studio Ghibli', '3D film']

Question: What is considered the first science fiction film?
     Answer : 2001: A Space Odyssey
     Sources: ['2001: A Space Odyssey', 'Star Wars (film)', 'The Matrix']

Question: Which movie introduced synchronized sound?
     Answer : phonofilm
     Sources: ['The Jazz Singer', 'Silent film', 'Sound film']

Question: Who founded Pixar?
     Answer : George Lucas
     Sources: ['Thomas Edison', 'George Lucas', 'Pixar']

Question: What year did Technicolor become popular?
     Answer : 1939
     Sources: ['History of animation', 'Casablanca (film)', 'The Jazz Singer']

Question: Which film is associated with the Rashomon effect?
     Answer : taurus
     Sources: ['3D film', 'History of film', 'History of film']

Question: What was

---
### Demo

In [30]:
def full_pipeline(question: str, top_k: int = 5, verbose: bool = True):
    print('=' * 65)
    print(f'Question: {question}')
    print('=' * 65)

    print(f'\nTop relevant documents BM25:')
    bm25_docs = bm25_retrieve(question, top_k=5)
    for i, d in enumerate(bm25_docs, 1):
        print(
            f'  [{i}] {d["title"]} | '
            f'BM25={d["bm25_score"]:.4f}'
        )

    print(f'\nTop relevant documents SBERT:')
    sbert_docs = sbert_retrieve_finetuned(question, top_k=5)
    for i, d in enumerate(sbert_docs, 1):
        print(
            f'  [{i}] {d["title"]} | '
            f'SBERT={d["sbert_score"]:.4f}'
        )

    top_passages = retrieve_passages(question, top_k=top_k)
    if verbose:
        print(f'\nTop-{top_k} passages retrieved (BM25 passage index):')
        for i, p in enumerate(top_passages, 1):
            print(f'  [{i}] {p["title"]} (score={p["score"]:.2f})')
            print(f'      {p["passage"][:180]}...')

    ext = extractive_answer(question, top_k_passages=top_k)
    print(f'\nExtractive Answer:')
    print(f'   Answer : {ext["answer"]}')
    print(f'   Source : {ext["source"]}')
    print(f'   Conf.  : {ext["score"]}')

    abs_ = abstractive_answer(question, top_k_passages=3)
    print(f'\nAbstractive Answer:')
    print(f'   Answer : {abs_["answer"]}')
    print(f'   Sources: {abs_["sources"]}')

    return {'extractive': ext, 'abstractive': abs_}

result = full_pipeline('Why is Seven Samurai considered influential?', top_k=5)
result2 = full_pipeline('How did international cinema influence Hollywood filmmaking?', top_k=5)
result3 = full_pipeline('What are the defining characteristics of film noir?', top_k=5)
result4 = full_pipeline('What are the origins of sound in cinema?', top_k=5)
result5 = full_pipeline('What genre is Steven Spielbergs Jaws?', top_k=5)

Question: Why is Seven Samurai considered influential?

Top relevant documents BM25:
  [1] Akira Kurosawa | BM25=1.9649
  [2] Stanley Kubrick | BM25=1.9468
  [3] Seven Samurai | BM25=1.9306
  [4] Hayao Miyazaki | BM25=1.8919
  [5] Taxi Driver | BM25=1.8397

Top relevant documents SBERT:
  [1] Seven Samurai | SBERT=0.3542
  [2] Akira Kurosawa | SBERT=0.3306
  [3] Federico Fellini | SBERT=0.3295
  [4] Studio Ghibli | SBERT=0.3231
  [5] Steven Spielberg | SBERT=0.2916

Top-5 passages retrieved (BM25 passage index):
  [1] John Ford (score=7.16)
      ntire meeting to ensure that demille remained in the guild  he then later offered his own resignation as part of the entire board to ensure that the guild did not break and allowed...
  [2] Intolerance (film) (score=7.05)
      ilm critics of the first half of the 20th century  believed that it was the only motion picture worthy of taking its place alongside beethoven s symphony no  5  michelangelo s sist...
  [3] The Matrix (score=6.90)
     

In [31]:
print(result)
print(result2)

{'extractive': {'answer': '', 'score': 4.5166332721710205, 'source': 'Luis Buñuel', 'passage': 'luis bun uel portole s  spanish    lwis  u  wel po to les   22 february 1900   29 july 1983  was a spanish filmmaker who worked in france  mexico  and spain  he has been widely considered by many film critics  historians  and directors to be one of the greatest and most influential filmmakers of all time  bun uel s works are known for their avant garde surrealism which was also infused with political commentary  often associated with the surrealist movement of the 1920s  bun uel s career spanned the 1920s through the 1970s  he collaborated with prolific surrealist painter salvador dali  on un chien andalou  1929  and l age d or  1930   both films are considered masterpieces of surrealist cinema  from 1947 to 1960  he honed his skills as a director in mexico  making grounded and human melodramas such as gran casino  1947   los olvidados  1950   and e l  1953   bun uel then transitioned into m